# Aquaplanet

SPEEDY T31L8 over a slab ocean with thermodynamic sea ice: JAX-ESM's own
quick start, built directly in Python -- the same construction
`docs/source/python_api.md` walks through, with the sea-ice component this
run also couples added on top (see the note below the build cell). The same
model, as one command:

```bash
python -m jem.main +configuration=aquaplanet-slab
```

Building it directly here, rather than composing that command's config and
calling `jem.runners.run(cfg)`, is the point of this notebook: every object
below is a plain Python class from `jem.components`, and nothing about it
depends on Hydra. The build cell passes `time_step=12` explicitly for the
same reason -- without it, `jcm.model.Model` would pick the physics' own
stable step (30 minutes) rather than the 12 minutes
`+configuration=aquaplanet-slab` actually runs at, which would make "the
same model, as one command" above false at the physics level.
`tests/unit/test_notebook_equivalence.py` checks the claim holds, `dt`
included. `02_aquaplanet_customized_initial_condition.ipynb` and
`03_aquaplanet_response_to_SST_perturbation_using_gradient.ipynb` instead use
`jem.configurations.load("aquaplanet-slab")` -- the recipe *door* onto this
same validated configuration -- because what they demonstrate (perturbing an
initial condition, differentiating through a run) has nothing to do with how
the model was assembled.

## Build it

In [ ]:
from pathlib import Path

from jem import plot

output_dir = (Path("output") / "01-01_aquaplanet").resolve()
output_dir.mkdir(parents=True, exist_ok=True)

This next cell is the direct construction: a plain `jcm.model.Model`
wrapped as a component, a `SlabGrid` taken from the atmosphere's own
horizontal grid, and a `Coupler` wiring them (plus the sea-ice component
below) with the standard exchange table. It is identical to
`docs/source/python_api.md`'s own quick-start block, up to and including
`print(repr(coupler))` -- pinned byte for byte by
`tests/unit/test_notebook_construction.py` -- except for the two lines
`SlabSeaiceModel` needs: the import, and its entry in `components`.
`python_api.md`'s own block stops at the atmosphere and the ocean (the
smallest coupled pair worth showing); this run is `aquaplanet-slab`, which
also couples the slab sea-ice model, so it is added here to keep this
notebook honest about what it actually runs -- an ocean alone has nowhere to
put the freeze/melt energy `SlabOceanModel` reports once its mixed layer
reaches the freezing point, and that energy would be silently dropped
without the sea-ice component.

In [ ]:
import jax_datetime as jdt
import jcm
from jcm.physics.speedy.speedy_coords import get_speedy_coords

from jem import Coupler, default_exchangers, run_chunked
from jem.components import JCMComponent, SlabOceanModel, SlabSeaiceModel
from jem.components.slab import SlabGrid

start_date = jdt.to_datetime("2000-01-01")
coupling_timestep = jdt.to_timedelta(1, "day")

# The JCM atmosphere: a plain jcm.model.Model, wrapped as a component.
# `time_step=12` (minutes) is the step `+configuration=aquaplanet-slab`
# actually runs at -- jax-gcm's `run/default.yaml`, composed at
# `atmosphere.run.time_step` -- and has to be given explicitly: with no
# `time_step`, `Model` instead picks the physics' own stable step (30
# minutes for SPEEDY T31L8), a materially different, faster-diffusing
# model than the one this notebook claims to reproduce.
atm_model = jcm.model.Model(
    coords=get_speedy_coords(), start_date=start_date, time_step=12
)
atm = JCMComponent(atm_model)

# Aquaplanet: the slab grid is built from the atmosphere's own horizontal grid,
# and with no fractional mask every cell is ocean.
grid = SlabGrid.from_coords(atm_model.coords.horizontal)

# An exchanger is the only place where components exchange information.
# `default_exchangers` is the standard wiring written down once — here, the
# atmosphere's surface heat flux drives the ocean and the ocean's SST comes
# back as the atmosphere's boundary condition — filtered to whichever of the
# standard components (`atm`, `ocn`, `lnd`, `seaice`) are present.
components = {
    "atm": atm,
    "ocn": SlabOceanModel(grid),
    "seaice": SlabSeaiceModel(grid, name="seaice"),
}
coupler = Coupler(
    components,
    default_exchangers(components),
    coupling_timestep=coupling_timestep,
    start_date=start_date,
)
print(repr(coupler))

## Run it

In [ ]:
result = run_chunked(
    coupler,
    total_time="30 days",
    chunk="30 days",
    output_dir=str(output_dir),
    subsample=3,       # 10 records out of 30 coupled days
    checkpoint_path=None,
)
result.steps_completed, [p.name for p in result.paths]

## What it wrote

One file per component per chunk, named after the coupled step its chunk starts on (`<component>-<first step>.nc`).

In [ ]:
atm_ds = plot.open_output(output_dir, "atm")
ocn_ds = plot.open_output(output_dir, "ocn")
seaice_ds = plot.open_output(output_dir, "seaice")
list(ocn_ds.data_vars)

## Plot

All three coupled components show up below: surface specific humidity (animated -- the field this example's top-level README entry is named after) and sea surface temperature from the atmosphere and ocean, and ice thickness from the sea-ice component.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# `level` is a sigma coordinate, surface-first, so selecting by its
# value (rather than `.isel(level=0)`) says so without relying on that
# ordering -- see "Output conventions" in
# docs/source/design/architecture.md.
humidity = atm_ds["specific_humidity"].sel(level=1.0, method="nearest")
plot.map_plot(humidity.isel(time=-1), ax=axes[0],
              title="Surface specific humidity [kg/kg]")

sst = ocn_ds["sea_surface_temperature"].isel(time=-1) - 273.15
plot.map_plot(sst, ax=axes[1], title="Sea surface temperature [°C]")

plot.map_plot(seaice_ds["ice_thickness"].isel(time=-1), ax=axes[2],
              title="Sea ice thickness [m]")
plt.tight_layout()

fig, ax = plt.subplots()
plot.area_mean(ocn_ds["sea_surface_temperature"]).plot(ax=ax)
ax.set_ylabel("Area-mean SST [K]")

# Animate the same field the static panel above shows, and the field
# the top-level README's gif is named after.
ani = plot.animate_map(humidity, title="Surface specific humidity [kg/kg]")
ani.save(output_dir / "specific_humidity.gif", writer="pillow")